## Introdução e Contexto:

A empresa Megaline quer desenvolver um modelo que possa analisar o comportamento do cliente e recomendar um dos planos mais recentes da Megaline: Smart ou Ultra.

Nosso conjunto de dados será os dados de comportamento dos **clientes que já mudaram para os novos planos**, com a feature "is_ultra" indicando se o plano atual é o Ultra (1), ou Smart(0).

O objetivo central é escolher qual o modelo capaz de executar a tarefa de classificação com a maior acurácia possível (neste projeto, o limite para acurácia é 0,75) e faremos a verificação da acurácia com o conjunto de dados de teste.

## Preparação do ambiente

### Importe as bibliotecas

In [12]:
# Configuração inicial
import pandas as pd
# Bibliotecas de visualização
import matplotlib.pyplot as plt
import seaborn as sns
#sklearn
from sklearn.metrics import accuracy_score, confusion_matrix, mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

### Definição das configurações de visualização.

In [2]:
# Variável Global para aleatoriedade
SEED = 42

# Ajuste das quantidade das colunas exibidas
pd.set_option('display.max_columns', 100)

# Estilo e tamanho padrão dos gráficos
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 7)

### Leitura dos Dados

In [7]:
df = pd.read_csv("users_behavior.csv")

print(df.head())
print('-'*50)
print(df.info())
print('-'*50)
print(df["is_ultra"].value_counts(normalize=True))

   calls  minutes  messages   mb_used  is_ultra
0   40.0   311.90      83.0  19915.42         0
1   85.0   516.75      56.0  22696.96         0
2   77.0   467.66      86.0  21060.45         0
3  106.0   745.53      81.0   8437.39         1
4   66.0   418.74       1.0  14502.75         0
--------------------------------------------------
<class 'pandas.DataFrame'>
RangeIndex: 3214 entries, 0 to 3213
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   calls     3214 non-null   float64
 1   minutes   3214 non-null   float64
 2   messages  3214 non-null   float64
 3   mb_used   3214 non-null   float64
 4   is_ultra  3214 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 125.7 KB
None
--------------------------------------------------
is_ultra
0    0.693528
1    0.306472
Name: proportion, dtype: float64


Como vimos acima, aparentemente não temos problemas com dados ausentes porque o `DataFrame` já foi preparado para o treinamento dos modelos.

Vale ressaltar neste ponto do projeto que a presença de usuários com plano ULTRA é 30%, então o desbalanceamento das classes neste problema de classificação é na razão de 70-30, mas isso não irá impactar o treinamento dos nossos modelos agora. 

## Preparação dos dados
Como o conjunto de dados de teste não existe, vamos dividir o conjunto de dados em Treinamento-Validação-Teste na proporção 3:1:1, isto é, 60% dos dados como Treinamento, 20% como validação, e 20% como dados de testes para as verificações finais. 

Vale ressaltar também, que já dividiremos de antemão as variáveis `features` e `target`, sendo a variável objetivo apenas a classificação `"is_ultra"` e as demais características permanecerão em features 

### Divisão das variáveis Feature-Target:

In [10]:
#Divisão do df features e objeto series target
features = df.drop(['is_ultra'], axis=1)
target = df['is_ultra']

### Divisão dos dados Treinamento-Validação:

In [33]:
#Atribuição Múltipla dividindo 60-40 inicialmente
features_train, features_valid1, target_train, target_valid1 = train_test_split(
    features,target, test_size = 0.4, random_state=SEED
)

### Divisão dos dados Validação-Teste:

In [37]:
#Atribuição Múltipla dividindo agora: 20% validação-20% teste
features_test, features_valid, target_test, target_valid = train_test_split(
    features_valid1,target_valid1, test_size = 0.5, random_state=SEED
)

0.2000622277535781


### Verificação das Proporções:

In [44]:
#Relatório final
print("Dados das features:")
print(f"Features de Treinamento: {len(features_train)*100 / len(df):.2f}% - Formato:{features_train.shape}")
print(f"Features de Validação: {len(features_valid)*100 / len(df):.2f}% - Formato:{features_valid.shape}")
print(f"Features de Teste: {len(features_test)*100 / len(df):.2f}% - Formato:{features_test.shape}")
print("\nDados de target:")
print(f"Target do Treinamento: {len(target_train)*100 / len(df):.2f}% - Formato:{target_train.shape}")
print(f"Target do Validação: {len(target_valid)*100 / len(df):.2f}% - Formato:{target_valid.shape}")
print(f"Target do Teste: {len(target_test)*100 / len(df):.2f}% - Formato:{target_test.shape}")

Dados das features:
Features de Treinamento: 59.99% - Formato:(1928, 4)
Features de Validação: 20.01% - Formato:(643, 4)
Features de Teste: 20.01% - Formato:(643, 4)

Dados de target:
Target do Treinamento: 59.99% - Formato:(1928,)
Target do Validação: 20.01% - Formato:(643,)
Target do Teste: 20.01% - Formato:(643,)


## Modelos para tarefa de Classificação:

Vamos a partir de agora nos concentrar em criar, treinar e avaliar como cada um dos modelos se comporta em relação ao `conjunto de dados de validação`. 
Para isso, testaremos 3 vias que se encaixam com a proposta de solução para o problema apresentado:

- `DecisionTreeClassifier`: árvore de decisão única com `max_depth`, `min_samples_split` e `min_samples_leaf` ajustáveis.

- `RandomFlorestClassifier`: floresta com árvore de decisão aleatórias, com `max_depth`, `n_estimators`,`min_samples_split` e `min_samples_leaf` ajustáveis.

- `LogisticRegression`: regressão a uma curva sigmoide com `solver` e `class_weigth` ajustáveis.

Três informações muito importantes devem ser reforçadas neste momento:

1. Todos os modelos serão treinados com os mesmos dados, para comparação justa.
2. Todos os modelos terão a mesma semente de aleatoriedade (SEED)
3. Neste caso, a métrica **ACURÁCIA** é o nosso **key_result** porque a classificação correta das duas classes é importante para a Megaline.


### DecisionTrees:

Para escolher o melhor modelo de árvore de decisão vamos criar ciclos de repetição alternando os hiperparâmetros para escolher o que possui a maior acurácia no conjunto de dados de validação. Escolhemos:
- `max_depth`: inteiro variando de 1 a 7, define a profundidade máxima de cada árvore

- `criterion`: será o de **gini** medindo a pureza de classe em cada nó, como o padrão do modelo

- `min_samples_split`: representa o número mínimo de observações que devem estar presentes em um nó antes que ele se divida. Testaremos com tipo float representando a fração de 1% a 5%, segundo a documentação oficial ele considerará o teto(min_samples_split * n_samples), se adaptando melhor ao tamanho do conjunto de dados de treinamento. De semelhante também testaremos parâmetros inteiros de 1 a 10, que neste caso serão mais permissivos.

- `min_samples_leaf`: representa o número mínimo de observações que um nó final (folha) deve ter. Testaremos de forma análoga ao feito acima.

In [ ]:
#Criando Loop para testar variações apenas em max_depth
#Iniciando o modelo:


In [ ]:
#Criando Loop para testar os valores tipo FLOAT
